<a href="https://colab.research.google.com/github/Davidson-Open-University/phy107-labs/blob/main/notebooks/PHY107-Lab7-Levers-Pulleys-and-Gears.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PHY 107 Lab 7: Levers, Pulleys and Gears

**Unit 7: Designing and building simple mechanical systems** · General Practical Physics I · Davidson Open University Global

**In this lab you will:**
- calculate the ideal mechanical advantage of a lever;
- find the actual mechanical advantage, velocity ratio and efficiency of a pulley system;
- use gear ratios to find speeds and torques;
- evaluate a simple machine from measurements.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the online lab and the examples, and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short analysis function and test it on data.
- **Part D – Your own data and reflection** (about 10 minutes): analyse readings you took in the online lab (or at home), and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `phy107-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

**Honest data:** keep your own readings separate from the example data, and label every simulated or supplied reading as such. Never invent readings.

*This lab is practice to help you learn and prepare for the unit assignment (lab report). It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False, dp=6):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), dp) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i, False, dp) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i, False, dp) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k, False, dp) + '=' + _norm(v, False, dp) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), dp) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('phy107|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False, spec.get('dp', 6))
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False), spec.get('dp', 6))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '7.1': {'h': 'af63a116843feecc', 'hint': 'Ideal MA = effort arm / load arm.', 'dp': 2},
    '7.2': {'h': '49b6df77b9de32c7', 'hint': 'Effort = load / MA.', 'dp': 1},
    '7.3': {'h': '3da764a89bdeee5f', 'hint': 'MA = load/effort; VR = effort distance / load distance; efficiency = work out / work in × 100.', 'ok': 'Friction and the weight of the pulleys make the efficiency less than 100%.', 'dp': 1},
    '7.4': {'h': '47a86572354322fa', 'hint': 'Speed ratio = driver teeth / driven teeth.', 'ok': 'Small driven gear = more speed, less torque.', 'dp': 2},
    '7.5': {'h': '77230dd993da3da4', 'hint': 'Output torque = input torque × driven teeth / driver teeth.', 'ok': 'Low gear on a bicycle: more turning force, less speed.', 'dp': 1},
    '7.6': {'h': '32e5d88a01855025', 'hint': 'MA = load/effort, VR = effort_dist/load_dist, efficiency = MA/VR × 100.', 'ok': 'You can now evaluate any simple machine from four measurements.', 'dp': 1, 'args': [(400, 150, 0.5, 2.0), (600, 220, 0.2, 0.6), (100, 60, 1.0, 2.0)]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
### 🔬 Online lab for this unit (free open educational resources)
- **[PhET Balancing Act](https://phet.colorado.edu/en/simulations/balancing-act)** – use the beam as a lever: balance a heavy load with a small effort further from the pivot.
- **[PhET Torque](https://phet.colorado.edu/en/simulations/torque)** – see how force and lever arm combine to turn a wheel.

Open the online lab in a new tab, try it for a few minutes, then come back here. **Label every reading you take there as "simulated".**

**Definitions.** Mechanical advantage MA = load ÷ effort. Velocity ratio VR = distance moved by effort ÷ distance moved by load. Efficiency = (work out ÷ work in) × 100% = (MA ÷ VR) × 100%. For gears, speed ratio = driver teeth ÷ driven teeth, and (ideally) torque is multiplied by driven teeth ÷ driver teeth.

**Safe home task:** count the teeth on the front chainring and the rear sprocket of a bicycle (or an okada, engine off). Do not put fingers near a moving chain.

In [ ]:
import numpy as np
effort_arm, load_arm = 1.2, 0.3
print('Ideal MA of this lever =', effort_arm / load_arm)

## Part B – Practise (about 20 minutes)
**B1.** A crowbar has an effort arm of 0.90 m and a load arm of 0.30 m. What is its ideal mechanical advantage?

In [ ]:
answer_7_1 = ...

In [ ]:
check('7.1', answer_7_1)

In [ ]:
answer_7_2 = ...   # effort (N) needed to lift a 600 N load with this ideal lever

In [ ]:
check('7.2', answer_7_2)

**B3. Pulley test.** A pulley system lifts a 400 N load by 0.50 m when an effort of 150 N is pulled through 2.0 m. Give `(MA, VR, efficiency %)`, rounded to 2, 1 and 1 decimal places.

In [ ]:
MA = ...
VR = ...
eff = ...
answer_7_3 = (MA, VR, eff)

In [ ]:
check('7.3', answer_7_3 if ... not in answer_7_3 else ...)

In [ ]:
answer_7_4 = ...   # a 48-tooth driver turns a 16-tooth driven gear: turns of the driven gear per turn of the driver

In [ ]:
check('7.4', answer_7_4)

In [ ]:
answer_7_5 = ...   # ideal output torque (N·m) when 30 N·m drives a 16-tooth gear meshed with a 48-tooth gear

In [ ]:
check('7.5', answer_7_5)

## Part C – Build a tool (about 20 minutes)
Complete `machine_report(load, effort, load_dist, effort_dist)`. It should return a tuple **(MA, VR, efficiency %)** rounded to 2, 2 and 1 decimal places.

In [ ]:
def machine_report(load, effort, load_dist, effort_dist):
    ...      # YOUR CODE

In [ ]:
check('7.6', machine_report)

## Part D – Your own data and reflection (about 10 minutes)
**D1. Prepares you for the Unit 7 assignment.** In Balancing Act, record three lever arrangements (load, load distance, effort, effort distance) and use your tool. If you counted bicycle gear teeth, calculate the speed ratio for two gear combinations. Label simulated and home data clearly.

In [ ]:
my_load, my_effort, my_load_dist, my_effort_dist = ..., ..., ..., ...

**D2. Reflect.**
1. Why can't a real machine have an efficiency of 100%?
2. A cyclist climbs a hill in a low gear. Explain why using torque and speed.
3. Suggest one improvement to raise the efficiency of a pulley system.

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `phy107-labs` repository → commit message *"Lab 7 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.